In [ ]:
!pip install -q -U adapters transformers>=4.41.0 datasets optuna evaluate scikit-learn accelerate tqdm pandas huggingface_hub torch torchao

In [ ]:
import torch
import pandas as pd
import numpy as np
import optuna
import evaluate
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, DefaultDataCollator
)
from adapters import (
    AdapterConfig, 
    AdapterTrainer, 
    setup_adapter_training
)
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import gc

In [ ]:
SEED = 15179996
model_id = "mistralai/Mistral-7B-v0.3"
db_path = "sqlite:///adapter_study.db"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def gpu_memory_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return 0

def compute_metrics(eval_pred):
    f1_metric = evaluate.load("f1")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    return f1_metric.compute(predictions=predictions, references=labels, average="weighted")

In [ ]:
def prepare_dataset(dataset_name="bdanko/sentiment140", train_size=5000, test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()

    # negative sentiment swapped to 1
    df['sentiment'] = df['sentiment'].replace(4, 1)

    train_df = df.sample(n=train_size, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=SEED)
    remaining_df = df.drop(train_df.index)

    test_df_neg = remaining_df[remaining_df['sentiment'] == 0].sample(n=test_size // 2, random_state=SEED)
    test_df_pos = remaining_df[remaining_df['sentiment'] == 1].sample(n=test_size // 2, random_state=SEED)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=SEED)

    return DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True))
    })

In [ ]:
dataset = prepare_dataset()
print(dataset)

In [ ]:
def tokenize_function(examples):
    texts = [f"Sentiment classification. Text: {text}\nSentiment: " for text in examples["text"]]
    tokenized = tokenizer(texts, truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = examples["sentiment"]
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
def get_base_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
    )
    model.config.pad_token_id = model.config.eos_token_id
    from adapters import init
    init(model)
    return model

In [ ]:
def adapter_objective(trial):
    reduction_factor = trial.suggest_categorical("reduction_factor", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)

    adapter_config = AdapterConfig(
        mh_adapter=True,
        output_adapter=True,
        reduction_factor=reduction_factor,
        non_linearity="relu",
        dropout=dropout
    )

    torch.cuda.reset_peak_memory_stats()
    trial_model = get_base_model()
    
    adapter_name = "sentiment_adapter_trial"
    trial_model.add_adapter(adapter_name, config=adapter_config)
    trial_model.train_adapter(adapter_name)
    trial_model.set_active_adapters(adapter_name)

    training_args = TrainingArguments(
        output_dir="./adapter_results",
        per_device_train_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        learning_rate=lr,
        eval_strategy="steps",
        eval_steps=50,
        logging_steps=50,
        bf16=True,
        report_to="none",
        save_strategy="no",
        disable_tqdm=True,
    )

    trainer = AdapterTrainer(
        model=trial_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    trial.report(eval_results["eval_f1"], step=1)
    if trial.should_prune():
        del trainer, trial_model
        gc.collect()
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned()

    trial.set_user_attr("peak_vram_gb", gpu_memory_gb())

    del trainer
    del trial_model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    return eval_results["eval_f1"]

In [ ]:
adapter_study = optuna.create_study(
    study_name="mistral_sentiment_adapter",
    storage=db_path,
    load_if_exists=True,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

adapter_study.optimize(adapter_objective, n_trials=10)

In [ ]:
def summarize_study(study_name, method_label):
    try:
        study = optuna.load_study(study_name=study_name, storage=db_path)
        df = study.trials_dataframe()
        df = df[df['state'] == 'COMPLETE'].copy()
        if 'user_attrs_peak_vram_gb' in df.columns:
            df['peak_vram_gb'] = df['user_attrs_peak_vram_gb']
        
        print(f"\n### Summary for {method_label} ({study_name}) ###")
        print(f"Best Trial F1: {study.best_value:.4f}")
        print(f"Best Parameters: {study.best_params}")
        
        cols_to_show = [col for col in df.columns if col.startswith('params_') or col in ['value', 'peak_vram_gb']]
        return df[cols_to_show].sort_values(by='value', ascending=False)
    except Exception as e:
        return f"Could not load {study_name}: {e}"

adapter_summary = summarize_study("mistral_sentiment_adapter", "Adapters")
display(adapter_summary.head())

In [ ]:
def train_best_adapter(study):
    best_params = study.best_params
    
    adapter_config = AdapterConfig(
        mh_adapter=True,
        output_adapter=True,
        reduction_factor=best_params['reduction_factor'],
        non_linearity="relu",
        dropout=best_params['dropout']
    )
    
    lr = best_params['lr']
    out_dir = "./best_adapter"
    hub_id = "bdanko/mistral-7b-sentiment-adapter"

    torch.cuda.reset_peak_memory_stats()
    model = get_base_model()
    
    adapter_name = "best_sentiment_adapter"
    model.add_adapter(adapter_name, config=adapter_config)
    model.train_adapter(adapter_name)
    model.set_active_adapters(adapter_name)

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=False,
        metric_for_best_model="f1",
        bf16=True,
        report_to="none",
        push_to_hub=True,
        hub_model_id=hub_id,
    )

    trainer = AdapterTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.model.save_pretrained(out_dir)
    
    final_metrics = trainer.evaluate()
    peak_vram = gpu_memory_gb()
    return trainer, final_metrics, peak_vram

In [ ]:
adapter_study = optuna.load_study(study_name="mistral_sentiment_adapter", storage=db_path)
best_adapter_trainer, best_adapter_metrics, adapter_vram = train_best_adapter(adapter_study)